[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/23_cross_attention_solution.ipynb)

# 🟡 Solution: Multi-Head Cross-Attention

*Attention & Transformers · Medium*

Reference implementation. Try it yourself in `23_cross_attention.ipynb` first.

---
Implement **multi-head cross-attention** as a `flax.nnx.Module`: the queries come
from one sequence, the keys and values from a different one, and the two have
unrelated lengths.

$$Q = x_q W_q,\quad K = x_{kv} W_k,\quad V = x_{kv} W_v,\qquad
\text{out} = \big[\operatorname{softmax}\!\big(\tfrac{Q_hK_h^\top}{\sqrt{d_h}} + M\big)V_h\big]_{h}W_o$$

with $M$ the additive form of the optional mask ($0$ where `mask` is `True`,
$-\infty$ elsewhere) and $[\cdot]_h$ the concatenation over heads.

### Signature
- `MultiHeadCrossAttention(d_model, num_heads, *, d_context=None, rngs: nnx.Rngs)`
- `d_context` defaults to `d_model`
- `__call__(x_q, x_kv, mask=None)` maps
  `(B, T_q, d_model)`, `(B, T_kv, d_context)` `->` `(B, T_q, d_model)`
- `mask` is boolean, broadcastable to `(B, H, T_q, T_kv)`; `True` = attend.
  A key-padding mask is `(B, 1, 1, T_kv)`

### Rules
- Subclass `nnx.Module`; no `nnx.MultiHeadAttention`, no
  `jax.nn.dot_product_attention`
- Parameters named exactly `w_q`, `w_k`, `w_v`, `w_o`, all `nnx.Param`, no biases:
  - `w_q`: `(d_model, d_model)`
  - `w_k`, `w_v`: `(d_context, d_model)`
  - `w_o`: `(d_model, d_model)`
- Initialise each with `jax.random.normal(rngs.params(), shape) / sqrt(fan_in)`
- **No causal mask** — the whole context is visible to every query
- `T_q` and `T_kv` are independent; nothing may assume they match

### Where this actually shows up
- **Encoder–decoder** (the original transformer, T5, Whisper): decoder tokens
  query the encoder's finished representation. $T_q$ is the tokens generated so
  far, $T_{kv}$ the source sentence or the audio frames.
- **Diffusion** (Stable Diffusion and descendants): the UNet's image latents are
  the queries and CLIP text embeddings are the keys/values. This is the *only*
  place the prompt enters the network — swap the K/V stream and you swap the
  prompt. It is also why `d_context` is a separate number: text encoders are 768
  or 1024 wide, UNet blocks are 320/640/1280.
- **Perceiver / Flamingo / DETR**: a small fixed array of learned queries reads
  an enormous input. Flamingo's Perceiver Resampler uses 64 latent queries;
  Perceiver ingests all $224^2 \approx 50\,000$ raw ImageNet pixels as keys;
  DETR decodes with 100 object queries over a convolutional feature map. Cost is
  $O(T_q T_{kv})$, not $O(T_{kv}^2)$ — cross-attention is how you get a
  fixed-cost bottleneck onto arbitrarily large inputs.

### The structural property worth naming in an interview
Cross-attention **cannot mix information across query positions**. Output $i$
depends on $x_q[i]$ and on all of $x_{kv}$, and on no other query. It is a
per-query lookup into a shared memory — batched, not sequential. That is why a
decoder block is always *self-attention, then cross-attention, then MLP*: the
self-attention layer is what lets query positions talk to each other, and
removing it leaves a model that can never build a representation spanning two
output tokens.

The second consequence is that the two streams get **separate lengths and
separate masks**. The causal mask belongs to self-attention; what cross-attention
needs is a key-padding mask over $T_{kv}$, shaped `(B, 1, 1, T_kv)` so it
broadcasts across heads and queries. Writing a `(T, T)` mask here is a bug that
only shows up when the two sequences happen to differ in length.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp
from flax import nnx


class MultiHeadCrossAttention(nnx.Module):
    def __init__(self, d_model: int, num_heads: int, *,
                 d_context: int | None = None, rngs: nnx.Rngs):
        if d_model % num_heads != 0:
            raise ValueError(f"d_model={d_model} is not divisible by num_heads={num_heads}")

        d_context = d_model if d_context is None else d_context
        self.d_model = d_model
        self.d_context = d_context
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        qs = 1.0 / jnp.sqrt(d_model)
        cs = 1.0 / jnp.sqrt(d_context)
        # K and V read the CONTEXT stream, so their fan-in is d_context.
        self.w_q = nnx.Param(jax.random.normal(rngs.params(), (d_model, d_model)) * qs)
        self.w_k = nnx.Param(jax.random.normal(rngs.params(), (d_context, d_model)) * cs)
        self.w_v = nnx.Param(jax.random.normal(rngs.params(), (d_context, d_model)) * cs)
        self.w_o = nnx.Param(jax.random.normal(rngs.params(), (d_model, d_model)) * qs)

    def _split_heads(self, x):
        """(B, T, d_model) -> (B, H, T, d_head)"""
        B, T, _ = x.shape
        return x.reshape(B, T, self.num_heads, self.d_head).transpose(0, 2, 1, 3)

    def __call__(self, x_q, x_kv, mask=None):
        B, T_q, _ = x_q.shape

        q = self._split_heads(x_q @ self.w_q)      # (B, H, T_q,  d_head)
        k = self._split_heads(x_kv @ self.w_k)     # (B, H, T_kv, d_head)
        v = self._split_heads(x_kv @ self.w_v)     # (B, H, T_kv, d_head)

        # Rectangular by construction: (B, H, T_q, T_kv).
        scores = (q @ jnp.swapaxes(k, -1, -2)) / jnp.sqrt(
            jnp.asarray(self.d_head, q.dtype)
        )
        if mask is not None:
            scores = jnp.where(mask, scores, jnp.asarray(-1e9, scores.dtype))

        weights = jax.nn.softmax(scores, axis=-1)
        out = (weights @ v).transpose(0, 2, 1, 3).reshape(B, T_q, self.d_model)
        return out @ self.w_o

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp
from flax import nnx

# A diffusion-style bridge: 256 image latents query 77 CLIP text tokens.
attn = MultiHeadCrossAttention(d_model=320, num_heads=8, d_context=768,
                               rngs=nnx.Rngs(params=0))
latents = jax.random.normal(jax.random.key(0), (1, 256, 320))
text = jax.random.normal(jax.random.key(1), (1, 77, 768))
print("w_q", attn.w_q.shape, " w_k", attn.w_k.shape, " out", attn(latents, text).shape)

# Query positions never interact: perturbing one query leaves the others alone.
small = MultiHeadCrossAttention(d_model=16, num_heads=2, rngs=nnx.Rngs(params=0))
xq = jax.random.normal(jax.random.key(2), (1, 4, 16))
xkv = jax.random.normal(jax.random.key(3), (1, 6, 16))
base = small(xq, xkv)
poked = small(xq.at[:, 2].set(99.0), xkv)
print("max change at query 0:", float(jnp.abs(base[:, 0] - poked[:, 0]).max()))
print("max change at query 2:", float(jnp.abs(base[:, 2] - poked[:, 2]).max()))

# A key-padding mask hiding the last two context tokens.
pad = jnp.array([[[[True, True, True, True, False, False]]]])   # (1, 1, 1, 6)
print("masked out:", small(xq, xkv, pad).shape)

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("cross_attention")